# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JasperOwen/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule in plain words:

Pages whose rankings have decreased should be reviewed for refresh first. Among these pages, those with high visibility should be prioritised as they will affect more people

Reason code: Ranking decline

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

import duckdb
from huggingface_hub import login

login(token=hf_token)

con = duckdb.connect()
con.sql("SET enable_http_metadata_cache=true;")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

file_path

'/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet'

In [3]:
page_performance_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    -- volume signal: clicks, both windows (SUM is safe here, no zero-quirk on clicks)
    SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS early_clicks,
    SUM(CASE WHEN report_date >  '2026-03-15' THEN gsc_clicks ELSE 0 END) AS late_clicks,

    -- visibility floor + CTR ingredient: impressions, both windows
    SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS early_impressions,
    SUM(CASE WHEN report_date >  '2026-03-15' THEN gsc_impressions ELSE 0 END) AS late_impressions,

    -- position signal: AVG, both windows, excluding the position=0 data quirk
    AVG(CASE WHEN report_date <= '2026-03-15' AND gsc_avg_position > 0
             THEN gsc_avg_position ELSE NULL END) AS early_avg_position,
    AVG(CASE WHEN report_date >  '2026-03-15' AND gsc_avg_position > 0
             THEN gsc_avg_position ELSE NULL END) AS late_avg_position

FROM read_parquet('{file_path}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""

page_performance = con.sql(page_performance_query).df()
page_performance.head()

,client_hash_id,content_hash_id,early_clicks,late_clicks,early_impressions,late_impressions,early_avg_position,late_avg_position
0,client_65de48885f4ef01b,content_5c80451459c29b4a,0.0,0.0,5.0,0.0,5.400000,NaN
1,client_65de48885f4ef01b,content_6b0149a80607dac3,2.0,10.0,465.0,734.0,8.056444,8.174629
2,client_65de48885f4ef01b,content_62673eea26c31c17,35.0,8.0,49386.0,7759.0,5.763388,7.866490
3,client_65de48885f4ef01b,content_872342e050545a12,0.0,0.0,39.0,0.0,6.538462,NaN
4,client_65de48885f4ef01b,content_4c185d1c173cd53d,6.0,3.0,156.0,122.0,11.026686,9.395247


In [4]:
import numpy as np
import pandas as pd

def position_tier(pos):
    if pd.isna(pos) or pos == 0:
      return np.nan

    if pos <= 3:
        return 1
    elif pos <= 10:
        return 2
    elif pos <= 20:
        return 3
    elif pos <= 50:
        return 4
    else:
        return 5

page_performance["early_position_tier"] = page_performance["early_avg_position"].apply(position_tier)
page_performance["late_position_tier"] = page_performance["late_avg_position"].apply(position_tier)
page_performance["tier_change"] = page_performance["late_position_tier"] - page_performance["early_position_tier"]

page_performance[["client_hash_id", "content_hash_id", "early_avg_position", "late_avg_position",
        "early_position_tier", "late_position_tier", "tier_change"]].head(10)

,client_hash_id,content_hash_id,early_avg_position,late_avg_position,early_position_tier,late_position_tier,tier_change
0,client_65de48885f4ef01b,content_5c80451459c29b4a,5.400000,NaN,2.0,NaN,NaN
1,client_65de48885f4ef01b,content_6b0149a80607dac3,8.056444,8.174629,2.0,2.0,0.0
2,client_65de48885f4ef01b,content_62673eea26c31c17,5.763388,7.866490,2.0,2.0,0.0
3,client_65de48885f4ef01b,content_872342e050545a12,6.538462,NaN,2.0,NaN,NaN
4,client_65de48885f4ef01b,content_4c185d1c173cd53d,11.026686,9.395247,3.0,2.0,-1.0
5,client_65de48885f4ef01b,content_bd07be40ea0d5f54,5.383152,33.655578,2.0,4.0,2.0
6,client_65de48885f4ef01b,content_40e28f4b41764012,6.327436,4.553223,2.0,2.0,0.0
7,client_65de48885f4ef01b,content_f8b204c8ce80dad5,7.697143,4.833333,2.0,2.0,0.0
8,client_65de48885f4ef01b,content_c943a83124c43e95,5.000000,NaN,2.0,NaN,NaN
9,client_c182d11e4862a37d,content_da76e1818babb4ce,11.348485,12.161290,3.0,3.0,0.0


In [5]:
print(page_performance[["tier_change"]].isna().sum())
print(len(page_performance))
print(f"tier_change missing: {34852/len(page_performance):.1%}")

tier_change    34852
dtype: int64
63856
tier_change missing: 54.6%


In [6]:
def position_change_bucket(change):
    if pd.isna(change):
        return np.nan
    elif change < 0:
        return "improved"
    elif change == 0:
        return "stable"
    else:
        return "declined"

page_performance["position_change"] = page_performance["tier_change"].apply(position_change_bucket)

In [7]:
position_signal_table = (
    page_performance[page_performance["position_change"].notna()]
    ["position_change"]
    .value_counts()
    .reset_index()
)

position_signal_table.columns = ["position_change", "n"]

position_signal_table

,position_change,n
0,stable,18507
1,declined,5683
2,improved,4814


Position change verdict: CONFIRMED

A meaningful subset of pages in the dataset are declining in ranking (19%), indicating that ranking decline is a reasonable signal for prioritising pages for refresh review.

In [8]:
def visibility_bucket(change):
    if pd.isna(change):
        return np.nan
    elif change <= 8:
        return "low"
    elif change > 8 and change <= 459:
        return "medium"
    else:
        return "high"

page_performance["visibility_bucket"] = page_performance["late_impressions"].apply(visibility_bucket)

In [9]:
visibility_table = (
    page_performance[page_performance["visibility_bucket"].notna()]
    ["visibility_bucket"]
    .value_counts()
    .reset_index()
)

visibility_table.columns = ["Visibility bucket", "n"]

visibility_table

,Visibility bucket,n
0,medium,31673
1,low,16231
2,high,15952


Visibility verdict: MIXED

By itself, the visibility of each page cannot tell us if a page is declining or not. However, when used alongside position change it can be used to help us decide which pages to prioritise for review, as pages with high visibility have greater search exposure, meaning they will benefit more from being reviewed for refresh.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
page_performance["visibility_score"] = page_performance["visibility_bucket"].map({
    "low": 1,
    "medium": 2,
    "high": 4
})

page_performance

,client_hash_id,content_hash_id,early_clicks,late_clicks,early_impressions,late_impressions,early_avg_position,late_avg_position,early_position_tier,late_position_tier,tier_change,position_change,visibility_bucket,visibility_score
0,client_65de48885f4ef01b,content_5c80451459c29b4a,0.0,0.0,5.0,0.0,5.400000,NaN,2.0,NaN,NaN,NaN,low,1
1,client_65de48885f4ef01b,content_6b0149a80607dac3,2.0,10.0,465.0,734.0,8.056444,8.174629,2.0,2.0,0.0,stable,high,4
2,client_65de48885f4ef01b,content_62673eea26c31c17,35.0,8.0,49386.0,7759.0,5.763388,7.866490,2.0,2.0,0.0,stable,high,4
3,client_65de48885f4ef01b,content_872342e050545a12,0.0,0.0,39.0,0.0,6.538462,NaN,2.0,NaN,NaN,NaN,low,1
4,client_65de48885f4ef01b,content_4c185d1c173cd53d,6.0,3.0,156.0,122.0,11.026686,9.395247,3.0,2.0,-1.0,improved,medium,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63851,client_20259bd6705d81d4,content_d41e42e293236000,0.0,0.0,0.0,7.0,NaN,24.428571,NaN,4.0,NaN,NaN,low,1
63852,client_20259bd6705d81d4,content_bef0c2e83e54fe21,0.0,0.0,0.0,45.0,NaN,24.400000,NaN,4.0,NaN,NaN,medium,2
63853,client_20259bd6705d81d4,content_5e0db3274a5b6e28,0.0,1.0,0.0,117.0,NaN,7.222222,NaN,2.0,NaN,NaN,medium,2
63854,client_20259bd6705d81d4,content_70db3d9a8ed58d6f,0.0,1.0,0.0,901.0,NaN,5.957825,NaN,2.0,NaN,NaN,high,4


In [11]:
def calculate_score(row):
    if row["tier_change"] > 0:
        return row["tier_change"] * row["visibility_score"]
    else:
        return 0

page_performance["page_score"] = page_performance.apply(calculate_score, axis=1)

page_performance

,client_hash_id,content_hash_id,early_clicks,late_clicks,early_impressions,late_impressions,early_avg_position,late_avg_position,early_position_tier,late_position_tier,tier_change,position_change,visibility_bucket,visibility_score,page_score
0,client_65de48885f4ef01b,content_5c80451459c29b4a,0.0,0.0,5.0,0.0,5.400000,NaN,2.0,NaN,NaN,NaN,low,1,0.0
1,client_65de48885f4ef01b,content_6b0149a80607dac3,2.0,10.0,465.0,734.0,8.056444,8.174629,2.0,2.0,0.0,stable,high,4,0.0
2,client_65de48885f4ef01b,content_62673eea26c31c17,35.0,8.0,49386.0,7759.0,5.763388,7.866490,2.0,2.0,0.0,stable,high,4,0.0
3,client_65de48885f4ef01b,content_872342e050545a12,0.0,0.0,39.0,0.0,6.538462,NaN,2.0,NaN,NaN,NaN,low,1,0.0
4,client_65de48885f4ef01b,content_4c185d1c173cd53d,6.0,3.0,156.0,122.0,11.026686,9.395247,3.0,2.0,-1.0,improved,medium,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63851,client_20259bd6705d81d4,content_d41e42e293236000,0.0,0.0,0.0,7.0,NaN,24.428571,NaN,4.0,NaN,NaN,low,1,0.0
63852,client_20259bd6705d81d4,content_bef0c2e83e54fe21,0.0,0.0,0.0,45.0,NaN,24.400000,NaN,4.0,NaN,NaN,medium,2,0.0
63853,client_20259bd6705d81d4,content_5e0db3274a5b6e28,0.0,1.0,0.0,117.0,NaN,7.222222,NaN,2.0,NaN,NaN,medium,2,0.0
63854,client_20259bd6705d81d4,content_70db3d9a8ed58d6f,0.0,1.0,0.0,901.0,NaN,5.957825,NaN,2.0,NaN,NaN,high,4,0.0


In [12]:
page_performance[page_performance["page_score"] > 0][
    ["tier_change", "visibility_bucket", "visibility_score", "page_score"]
].head(10)

,tier_change,visibility_bucket,visibility_score,page_score
5,2.0,medium,2,4.0
12,1.0,medium,2,2.0
15,1.0,high,4,4.0
23,1.0,high,4,4.0
33,3.0,medium,2,6.0
40,1.0,high,4,4.0
73,1.0,medium,2,2.0
86,1.0,high,4,4.0
110,1.0,medium,2,2.0
117,1.0,medium,2,2.0


In [13]:
page_performance["reason_code"] = "ranking_decline"
page_performance["action"] = "review for refresh"


page_performance

,client_hash_id,content_hash_id,early_clicks,late_clicks,early_impressions,late_impressions,early_avg_position,late_avg_position,early_position_tier,late_position_tier,tier_change,position_change,visibility_bucket,visibility_score,page_score,reason_code,action
0,client_65de48885f4ef01b,content_5c80451459c29b4a,0.0,0.0,5.0,0.0,5.400000,NaN,2.0,NaN,NaN,NaN,low,1,0.0,ranking_decline,review for refresh
1,client_65de48885f4ef01b,content_6b0149a80607dac3,2.0,10.0,465.0,734.0,8.056444,8.174629,2.0,2.0,0.0,stable,high,4,0.0,ranking_decline,review for refresh
2,client_65de48885f4ef01b,content_62673eea26c31c17,35.0,8.0,49386.0,7759.0,5.763388,7.866490,2.0,2.0,0.0,stable,high,4,0.0,ranking_decline,review for refresh
3,client_65de48885f4ef01b,content_872342e050545a12,0.0,0.0,39.0,0.0,6.538462,NaN,2.0,NaN,NaN,NaN,low,1,0.0,ranking_decline,review for refresh
4,client_65de48885f4ef01b,content_4c185d1c173cd53d,6.0,3.0,156.0,122.0,11.026686,9.395247,3.0,2.0,-1.0,improved,medium,2,0.0,ranking_decline,review for refresh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63851,client_20259bd6705d81d4,content_d41e42e293236000,0.0,0.0,0.0,7.0,NaN,24.428571,NaN,4.0,NaN,NaN,low,1,0.0,ranking_decline,review for refresh
63852,client_20259bd6705d81d4,content_bef0c2e83e54fe21,0.0,0.0,0.0,45.0,NaN,24.400000,NaN,4.0,NaN,NaN,medium,2,0.0,ranking_decline,review for refresh
63853,client_20259bd6705d81d4,content_5e0db3274a5b6e28,0.0,1.0,0.0,117.0,NaN,7.222222,NaN,2.0,NaN,NaN,medium,2,0.0,ranking_decline,review for refresh
63854,client_20259bd6705d81d4,content_70db3d9a8ed58d6f,0.0,1.0,0.0,901.0,NaN,5.957825,NaN,2.0,NaN,NaN,high,4,0.0,ranking_decline,review for refresh


In [14]:
ranked_queue = page_performance[page_performance["page_score"] > 0][["client_hash_id", "content_hash_id", "page_score", "reason_code","action"]]
ranked_queue = ranked_queue.sort_values("page_score", ascending=False)
ranked_queue = ranked_queue.reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

ranked_queue

,client_hash_id,content_hash_id,page_score,reason_code,action
0,client_20259bd6705d81d4,content_b82e798f66048fe8,12.0,ranking_decline,review for refresh
1,client_20259bd6705d81d4,content_2722763d7f417e61,12.0,ranking_decline,review for refresh
2,client_fef1a8f436438636,content_1857068cc039c04a,12.0,ranking_decline,review for refresh
3,client_23a62021009f63c4,content_c74d93889f83b620,12.0,ranking_decline,review for refresh
4,client_23a62021009f63c4,content_fad516a4d7587469,12.0,ranking_decline,review for refresh
...,...,...,...,...,...
5678,client_e5c2aa26a8598242,content_bfb7030fd71a3102,1.0,ranking_decline,review for refresh
5679,client_fef1a8f436438636,content_488e4b6f20b92104,1.0,ranking_decline,review for refresh
5680,client_fef1a8f436438636,content_45a79cef887e0478,1.0,ranking_decline,review for refresh
5681,client_fef1a8f436438636,content_38ad8b6a3f089eac,1.0,ranking_decline,review for refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Each of the first 16 rows below has the highest page score possible. This is due to having the largest combination of ranking decline and visibility. I confirmed this by observing the ranking decline and visibility level of each page, meaning I am confident that the pages are ranked according to my baseline rule.However, the rankings presented in the queue could be wrong due to the decline in ranking being caused by reasons unrelated to the content of each page. For example, it could be a seasonal issue or a technical issue with the webpage.

The last 4 pages below have a lower page score. This is an indicator of either a smaller ranking decline with higher visibility or a middling ranking decline with middling visibility. I confirmed this by observing the ranking decline and visibility level of each page, meaning I am confident that the pages are ranked according to my baseline rule. However, these rankings could still be wrong due to similar reasons as the higher ranking pages such as seasonality or technical issues.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_20 = ranked_queue.head(20)
top_20

,client_hash_id,content_hash_id,page_score,reason_code,action
0,client_20259bd6705d81d4,content_b82e798f66048fe8,12.0,ranking_decline,review for refresh
1,client_20259bd6705d81d4,content_2722763d7f417e61,12.0,ranking_decline,review for refresh
2,client_fef1a8f436438636,content_1857068cc039c04a,12.0,ranking_decline,review for refresh
3,client_23a62021009f63c4,content_c74d93889f83b620,12.0,ranking_decline,review for refresh
4,client_23a62021009f63c4,content_fad516a4d7587469,12.0,ranking_decline,review for refresh
5,client_23a62021009f63c4,content_e3231c64697a6ea9,12.0,ranking_decline,review for refresh
6,client_fef1a8f436438636,content_5c968313d4668c77,12.0,ranking_decline,review for refresh
7,client_23a62021009f63c4,content_23e1f7b8319bc0e5,12.0,ranking_decline,review for refresh
8,client_23a62021009f63c4,content_7e53813b868da196,12.0,ranking_decline,review for refresh
9,client_20259bd6705d81d4,content_c4646f7e1cd1c9db,12.0,ranking_decline,review for refresh


In [16]:
# Full table used to confirm ranked queue results.
# Exact ordering differs from ranked queue for each score.
# For example, the pages with a page score of 12 are in a different order

page_performance = page_performance.sort_values("page_score", ascending=False)
page_performance = page_performance.reset_index(drop=True)

page_performance.head(20)

,client_hash_id,content_hash_id,early_clicks,late_clicks,early_impressions,late_impressions,early_avg_position,late_avg_position,early_position_tier,late_position_tier,tier_change,position_change,visibility_bucket,visibility_score,page_score,reason_code,action
0,client_23a62021009f63c4,content_7e53813b868da196,4.0,3.0,1529.0,4637.0,2.408639,34.563390,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
1,client_23a62021009f63c4,content_c74d93889f83b620,3.0,5.0,479.0,1953.0,2.930923,25.986976,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
2,client_23a62021009f63c4,content_23e1f7b8319bc0e5,5.0,0.0,544.0,3067.0,2.342278,22.170569,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
3,client_20259bd6705d81d4,content_2722763d7f417e61,7.0,3.0,538.0,538.0,2.784379,20.076507,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
4,client_fef1a8f436438636,content_1857068cc039c04a,1.0,4.0,131.0,1442.0,2.639472,26.325659,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
5,client_e5c2aa26a8598242,content_5724001a3c6298b7,4.0,16.0,1175.0,5539.0,2.880471,22.892097,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
6,client_23a62021009f63c4,content_55c33a4c7e6624b1,4.0,16.0,279.0,5453.0,2.886367,31.293120,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
7,client_20259bd6705d81d4,content_ec13486e79bca37b,4.0,2.0,340.0,978.0,2.716239,27.724314,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
8,client_23a62021009f63c4,content_fad516a4d7587469,0.0,0.0,355.0,899.0,1.807150,30.752809,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
9,client_23a62021009f63c4,content_c7540a12b1036697,1.0,6.0,126.0,1244.0,0.669547,27.662945,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

While my top 20 results appear to rank the pages effectively, with each page having a tier change of 3, lower down in the queue a page with high visibility but a small ranking decline could potentially be prioritised higher than pages with greater ranking decline. These pages with small decline may not be declining enough to justify needing a review, potentially causing the rankings to be incorrect.

My baseline rule only uses historical data available in the dataset to determine the page's ranking and visibility level during March 2026. It does not use any labels or future data, so there is no data leakage.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.